## Set-up

In [2]:
# Import core and specialist libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tqdm as notebook_tqdm

# Preprocessing, metrics, clustering, PCA, t-SNE, and UMAP
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import hdbscan, umap

# Configure plotting styles and disable warnings
# import warnings
# warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

print("Setup complete.")

Setup complete.


Import & parse geojson dataframe:

In [10]:
#open file
with open("business-licences.geojson") as f:
    #load geojson_data using json parser
    geojson_data = json.load(f)

#print total features count
print("# of features:", len(geojson_data['features']))
#print formatted JSON representation of the first feature
print()
print(json.dumps(geojson_data['features'][3], indent=2))

# of features: 204107

{
  "type": "Feature",
  "geometry": {
    "coordinates": [
      -123.118192793095,
      49.2877342016388
    ],
    "type": "Point"
  },
  "properties": {
    "folderyear": "24",
    "licencersn": "4518848",
    "licencenumber": "24-138567",
    "licencerevisionnumber": "10",
    "businessname": "Baron Global Financial Canada Ltd",
    "businesstradename": null,
    "status": "Issued",
    "issueddate": "2023-12-11T10:12:12-08:00",
    "expireddate": "2024-12-31",
    "businesstype": "Consulting and Management Services",
    "businesssubtype": null,
    "unit": "2250",
    "unittype": "Unit",
    "house": "1055",
    "street": "W HASTINGS ST",
    "city": "Vancouver",
    "province": "BC",
    "country": "CA",
    "postalcode": "V6E 2E9",
    "localarea": "Downtown",
    "numberofemployees": 5.0,
    "feepaid": null,
    "extractdate": "2026-07-01T02:32:17-07:00",
    "geo_point_2d": {
      "lon": -123.118192793095,
      "lat": 49.2877342016388
    }
  }
}


In [ ]:
#flatten GeoJSON into a standard 2D pandas DataFrame 
rows = []
for feature in geojson_data["features"]:
    props = feature['properties']
    #if there are no geo coords, default to empty array
    geo = props.get('geo_point_2d') or {}
    #extract
    lat = geo.get('lat')
    lon = geo.get('lon')

    #incase the geo2d doesn't have lat and lon but geom does:
    if lat is None or lon is None:
        geom = feature.get('geometry')
        #if geom is dictionary then extracts coords
        if isinstance(geom, dict):
            coords = geom.get('coordinates')
            # coords is a list/tuple with at least 2 elements
            if isinstance(coords, (list, tuple)) and len(coords) >= 2:
                lon, lat = coords[0], coords[1]  # GeoJSON stores [longitude, latitude]
        
    #append
    rows.append({**props, 'latitude': lat, 'longitude': lon})

df = pd.DataFrame(rows)

# Inspect
print("shape:", df.shape)
df.head()

shape: (204107, 27)


,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,latitude,longitude
0,24,4518841,24-138560,10,Nobl Collective Ltd,NaN,Issued,2023-12-28T13:34:28-08:00,2024-12-31,Consulting and Management Services,...,CA,NaN,Kitsilano,1.0,NaN,2026-07-01T02:32:17-07:00,NaN,None,NaN,NaN
1,24,4518843,24-138562,10,645064 BC Ltd,Trade Exchange Canada,Issued,2023-11-20T13:40:18-08:00,2024-12-31,Business Support Services,...,CA,NaN,Kitsilano,1.0,NaN,2026-07-01T02:32:17-07:00,NaN,None,NaN,NaN
2,24,4518844,24-138563,10,(Louise Turgeon),Turgeon Business Consulting,Issued,2023-11-21T14:30:07-08:00,2024-12-31,Consulting and Management Services,...,CA,NaN,Downtown,1.0,NaN,2026-07-01T02:32:17-07:00,NaN,None,NaN,NaN
3,24,4518848,24-138567,10,Baron Global Financial Canada Ltd,NaN,Issued,2023-12-11T10:12:12-08:00,2024-12-31,Consulting and Management Services,...,CA,V6E 2E9,Downtown,5.0,NaN,2026-07-01T02:32:17-07:00,NaN,"{'lon': -123.118192793095, 'lat': 49.287734201...",49.287734,-123.118193
4,24,4518856,24-138576,10,Jennifer D S Dezell (Jennifer Dezell),Dentons Canada LLP,Issued,2023-12-08T16:34:17-08:00,2024-12-31,Legal Services,...,CA,V6C 3R8,Downtown,217.0,NaN,2026-07-01T02:32:17-07:00,NaN,"{'lon': -123.113252488858, 'lat': 49.286652613...",49.286653,-123.113252


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 204107 entries, 0 to 204106
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   folderyear             204107 non-null  str    
 1   licencersn             204107 non-null  str    
 2   licencenumber          204107 non-null  str    
 3   licencerevisionnumber  204107 non-null  str    
 4   businessname           190635 non-null  str    
 5   businesstradename      77307 non-null   str    
 6   status                 204107 non-null  str    
 7   issueddate             175296 non-null  str    
 8   expireddate            175320 non-null  str    
 9   businesstype           204107 non-null  str    
 10  businesssubtype        21518 non-null   str    
 11  unit                   48914 non-null   str    
 12  unittype               48650 non-null   str    
 13  house                  109704 non-null  str    
 14  street                 109721 non-null  str    

In [26]:
df.describe()

,numberofemployees,feepaid,geom,latitude,longitude
count,204107.000000,128649.000000,0.0,102566.000000,102566.000000
mean,10.027549,517.069219,NaN,49.265861,-123.112127
std,72.798081,1110.394696,NaN,0.021525,0.031856
min,0.000000,2.000000,NaN,49.200531,-123.219197
25%,0.000000,207.000000,NaN,49.260353,-123.127872
50%,1.000000,277.000000,NaN,49.269686,-123.117238
75%,4.000000,405.000000,NaN,49.282621,-123.098740
max,5876.000000,63722.000000,NaN,49.295631,-123.024016


## A1 — Data acquisition and cleaning

### Data-cleaning

1. Records with No Geographic Coordinates: `geom` `geo_point_2d` `geometry` & `latitude` `longitude` 

We can't apply K-Means or DBScan on locations without coordinations, therefore these should be removed

In [32]:
# drop rows with missing latitude & longitude 
df_spatial = df.dropna(subset=['latitude', 'longitude']).copy()
print("length before:", len(df['longitude']))
print("length after:", len(df_spatial['longitude']))

length before: 204107
length after: 102566
